<a href="https://colab.research.google.com/github/heraclides/Batman/blob/master/criando_transcri%C3%A7%C3%A3o_de_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalando o reconhecimento de Voz do Google

!pip install -q SpeechRecognition

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 34.5 MB/s eta 0:00:00


In [ ]:
# instalando o tradutor
!pip install -q deep-translator
import speech_recognition as sr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.0 MB/s eta 0:00:00


In [ ]:
from base64 import b64decode
from google.colab import output
import speech_recognition as sr

# O Colab roda no navegador, então o microfone é acessado via JavaScript
def gravar_audio(segundos=5):
    codigo_js = """
    (async () => {
      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();

      //cria uma div para mostrar a mensagem
      const status = document.createElement("div");

        status.style.fontSize = "28px";
        status.style.fontWeight = "bold";
        status.style.margin = "20px";

        document.body.appendChild(status);

      status.innerHTML = "Fale algo. O programa grava 5 segundos e transcreve."
      await new Promise(r => setTimeout(r, SEGUNDOS * 1000));
      recorder.stop();

      //remove a mensagem
      status.remove();
      await new Promise(r => recorder.onstop = r);
      const blob = new Blob(chunks);
      const reader = new FileReader();
      const dataUrl = await new Promise(r => {
        reader.onload = () => r(reader.result);
        reader.readAsDataURL(blob);
      });
      return dataUrl;
    })()
    """.replace("SEGUNDOS", str(segundos))

    data_url = output.eval_js(codigo_js)      # executa o JS no navegador
    return b64decode(data_url.split(",")[1])  # converte base64 em bytes

In [ ]:
#print("=== Transcrição contínua ===")
#print("Fale algo. O programa grava 5 segundos e transcreve.")
audio_bytes = gravar_audio(10)
with open("audio.webm", "wb") as f:
    f.write(audio_bytes)
print("Áudio gravado com sucesso!")

Áudio gravado com sucesso!


In [ ]:
# verificando o tamanho do arquivo de audio
import os #biblioteca nativa do python que permite acessa arquivos e pastas

tamanho_bytes = os.path.getsize("audio.webm")
# os.path acessa o caminho do arquivo
# getsize pergunta; Qual tamanho desse arquivo?
tamanho_kb = tamanho_bytes / 1024
tamanho_mb = tamanho_kb / 1024

print(f"📦 Tamanho: {tamanho_bytes} bytes")
print(f"📦 Tamanho: {tamanho_kb:.2f} KB")
print(f"📦 Tamanho: {tamanho_mb:.2f} MB")

📦 Tamanho: 160652 bytes
📦 Tamanho: 156.89 KB
📦 Tamanho: 0.15 MB


In [ ]:
# converte o arquivo WEBM para WAV
!ffmpeg -y -i audio.webm audio.wav -loglevel quiet
print("Conversão concluída!")

Conversão concluída!


In [ ]:
# verifica se o audio realmente tem 5s
import wave # biblioteca especifica para trabalhar com arquivos WAV

with wave.open("audio.wav", "rb") as arquivo:
  # wave.open abre o arquivo .wav
  #rb -> r de reed -> leitura e b de binary -> binario

    frames = arquivo.getnframes() #obtem a quantidades de frames do audio
    taxa = arquivo.getframerate() #obtem quantos frames exitem por segundo

    duracao = frames / float(taxa)

print(f"⏱️ Duração do áudio: {duracao:.2f} segundos")

⏱️ Duração do áudio: 9.96 segundos


In [ ]:
recognizer = sr.Recognizer()

with sr.AudioFile("audio.wav") as fonte:
    audio = recognizer.record(fonte)

try:
    texto = recognizer.recognize_google(audio, language="pt-BR")
    print("Transcrição:", texto)
except sr.UnknownValueError:
    print("Não entendi o áudio. Fale mais devagar ou mais perto do microfone.")
except sr.RequestError as e:
    print("Erro ao acessar a API do Google:", e)

Transcrição: boa noite tudo bom


In [ ]:
# Criando loop para falar varias vezes

print("=== Transcrição contínua ===")
print("Fale algo. O programa grava 5 segundos e transcreve.")
print("Para encerrar, digite: sair\n")

while True:
    # 1) Grava 5 segundos
    print("🎤 Gravando 5 segundos... fale agora!")
    audio_bytes = gravar_audio(5)

    # 2) Salva e converte
    with open("audio.webm", "wb") as f:
        f.write(audio_bytes)
    !ffmpeg -y -i audio.webm audio.wav -loglevel quiet

    # 3) Transcreve
    with sr.AudioFile("audio.wav") as fonte:
        audio = recognizer.record(fonte)

    try:
        texto = recognizer.recognize_google(audio, language="pt-BR")
        print("📝 Transcrição:", texto)
    except sr.UnknownValueError:
        print("😕 Não entendi. Tente falar mais devagar.")
    except sr.RequestError as e:
        print("⚠️ Erro na API:", e)

    # 4) Pergunta se quer continuar
    comando = input("\nDigite ENTER para gravar de novo, ou 'sair' para encerrar: ").strip().lower()
    if comando == "sair":
        print("👋 Encerrando o programa.")
        break

=== Transcrição contínua ===
Fale algo. O programa grava 5 segundos e transcreve.
Para encerrar, digite: sair

🎤 Gravando 5 segundos... fale agora!
📝 Transcrição: fala fala fala fala fala

Digite ENTER para gravar de novo, ou 'sair' para encerrar: sair
👋 Encerrando o programa.


In [ ]:
#Código JS com contador recressivo de gravação

from google.colab import output
from IPython.display import Javascript, display
import base64
import time


def gravar_audio(segundos=15):

    js = Javascript(f"""
    async function gravarAudio() {{

        // Solicita acesso ao microfone
        const stream = await navigator.mediaDevices.getUserMedia({{
            audio: true
        }});

        // Cria o gravador
        const recorder = new MediaRecorder(stream);

        const chunks = [];

        recorder.ondataavailable = event => {{
            chunks.push(event.data);
        }};

        // -----------------------------------
        // CONTAGEM ANTES DA GRAVAÇÃO
        // -----------------------------------

        const status = document.createElement("div");

        status.style.fontSize = "28px";
        status.style.fontWeight = "bold";
        status.style.margin = "20px";

        document.body.appendChild(status);

        status.innerHTML = "🎙️ Preparando microfone...";

        await new Promise(
            r => setTimeout(r, 1000)
        );

        for (let i = 3; i > 0; i--) {{

            status.innerHTML =
                "Prepare-se... " + i;

            await new Promise(
                r => setTimeout(r, 1000)
            );
        }}


        // -----------------------------------
        // INICIA A GRAVAÇÃO
        // -----------------------------------

        recorder.start();


        // -----------------------------------
        // CONTAGEM DURANTE A GRAVAÇÃO
        // -----------------------------------

        for (let i = {segundos}; i > 0; i--) {{

            status.innerHTML =
                "🔴 GRAVANDO... " +
                i +
                " segundos";

            await new Promise(
                r => setTimeout(r, 1000)
            );
        }}


        // -----------------------------------
        // PARA A GRAVAÇÃO
        // -----------------------------------

        recorder.stop();

        status.innerHTML =
            "⏹️ Gravação finalizada!";


        // Aguarda o gravador finalizar
        await new Promise(resolve => {{

            recorder.onstop = resolve;

        }});


        // Desliga o microfone
        stream.getTracks().forEach(
            track => track.stop()
        );


        // -----------------------------------
        // CONVERTE O ÁUDIO
        // -----------------------------------

        const blob = new Blob(
            chunks,
            {{type: "audio/webm"}}
        );

        const reader = new FileReader();

        const dataUrl = await new Promise(resolve => {{

            reader.onloadend = () =>
                resolve(reader.result);

            reader.readAsDataURL(blob);

        }});


        // Remove a mensagem depois de 1 segundo
        setTimeout(() => {{
            status.remove();
        }}, 1000);


        return dataUrl;
    }}

    gravarAudio();
    """)

    # Executa o JavaScript
    resultado = output.eval_js(
        js.data
    )

    # Remove o cabeçalho base64
    audio_base64 = resultado.split(",")[1]

    # Converte para bytes
    audio_bytes = base64.b64decode(
        audio_base64
    )

    return audio_bytes

In [ ]:
import speech_recognition as sr

recognizer = sr.Recognizer()
tempo = 20

print("===================================")
print("     TRANSCRIÇÃO CONTÍNUA")
print("===================================")

print(f"\nO programa grava {tempo} segundos.")
print("Aguarde a contagem 3, 2, 1.")
print("Depois comece a falar.\n")


while True:

    # ===================================
    # 1. GRAVAR
    # ===================================

    audio_bytes = gravar_audio(tempo)


    # ===================================
    # 2. SALVAR WEBM
    # ===================================

    with open("audio.webm", "wb") as f:

        f.write(audio_bytes)


    # ===================================
    # 3. CONVERTER PARA WAV
    # ===================================

    !ffmpeg -y -i audio.webm audio.wav -loglevel quiet


    # ===================================
    # 4. CARREGAR O ÁUDIO
    # ===================================

    with sr.AudioFile("audio.wav") as fonte:

        audio = recognizer.record(fonte)


    # ===================================
    # 5. TRANSCREVER
    # ===================================

    try:

        texto = recognizer.recognize_google(
            audio,
            language="pt-BR"
        )

        print("\n📝 TRANSCRIÇÃO:")
        print(texto)


    except sr.UnknownValueError:

        print(
            "\n😕 Não consegui entender."
        )


    except sr.RequestError as e:

        print(
            "\n⚠️ Erro no serviço:",
            e
        )


    # ===================================
    # 6. CONTINUAR?
    # ===================================

    comando = input(
        "\nPressione ENTER para falar novamente "
        "ou digite 'sair': "
    )

    if comando.strip().lower() == "sair":

        print(
            "\n👋 Programa encerrado."
        )

        break

In [ ]:
recognizer = sr.Recognizer()

In [ ]:
import time

print("=== Transcrição contínua ===")
print("O programa grava 10 segundos e transcreve.")
print("Para encerrar, digite: sair\n")

while True:

    # ---------------------------------
    # 1. Preparação
    # ---------------------------------

    print("\n🎙️ Preparando para gravar...")

    for i in range(3, 0, -1):
        print(i)
        time.sleep(1)

    print("🔴 AGORA! Pode falar.")

    # ---------------------------------
    # 2. Gravar 10 segundos
    # ---------------------------------

    audio_bytes = gravar_audio(10)

    print("⏹️ Gravação finalizada.")

    # ---------------------------------
    # 3. Salvar arquivo
    # ---------------------------------

    with open("audio.webm", "wb") as f:
        f.write(audio_bytes)

    # Converter WEBM → WAV
    !ffmpeg -y -i audio.webm audio.wav -loglevel quiet

    # ---------------------------------
    # 4. Ler áudio
    # ---------------------------------

    with sr.AudioFile("audio.wav") as fonte:
        audio = recognizer.record(fonte)

    # ---------------------------------
    # 5. Transcrever
    # ---------------------------------

    try:

        texto = recognizer.recognize_google(
            audio,
            language="pt-BR"
        )

        print("\n📝 Transcrição:")
        print(texto)

    except sr.UnknownValueError:

        print(
            "😕 Não entendi. "
            "Tente falar mais devagar."
        )

    except sr.RequestError as e:

        print(
            "⚠️ Erro na API:",
            e
        )

    # ---------------------------------
    # 6. Continuar ou sair
    # ---------------------------------

    comando = input(
        "\nPressione ENTER para gravar novamente "
        "ou digite 'sair': "
    ).strip().lower()

    if comando == "sair":

        print("👋 Encerrando o programa.")

        break

In [ ]:
# importando arquivo para gravação

from google.colab import files

arquivo_enviado = files.upload()
nome_arquivo = list(arquivo_enviado.keys())[0]
print("Arquivo recebido:", nome_arquivo)

In [ ]:
# @title
recognizer = sr.Recognizer()

with sr.AudioFile(nome_arquivo) as fonte:
    audio = recognizer.record(fonte)
idiomas = ["pt-BR", "en-US", "es-ES", "fr-FR", "de-DE"]

melhor_texto = ""
melhor_idioma = ""
melhor_confianca = 0.0

for idioma in idiomas:
    try:
        resultado = recognizer.recognize_google(audio, language=idioma, show_all=True)
        if resultado:
            alternativa = resultado["alternative"][0]
            texto = alternativa["transcript"]
            confianca = alternativa.get("confidence", 0)
            print(f"{idioma}: confiança {confianca:.2f} → {texto}")
            if confianca > melhor_confianca:
                melhor_confianca = confianca
                melhor_texto = texto
                melhor_idioma = idioma
    except sr.UnknownValueError:
        print(f"{idioma}: não entendi o áudio")
    except sr.RequestError as e:
        print(f"{idioma}: erro de conexão: {e}")

print("\n=== Melhor resultado ===")
print("Idioma detectado:", melhor_idioma)
print("Transcrição:", melhor_texto)

try:
    texto = recognizer.recognize_google(audio, language= melhor_idioma)
    print("Transcrição:", texto)
except sr.UnknownValueError:
    print("Não entendi o áudio. Fale mais devagar ou mais perto do microfone.")
except sr.RequestError as e:
    print("Erro ao acessar a API do Google:", e)

In [ ]:
import speech_recognition as sr

In [ ]:
recognizer = sr.Recognizer()

with sr.AudioFile(nome_arquivo) as fonte:
    audio = recognizer.record(fonte)



try:
    texto = recognizer.recognize_google(audio, language="pt-Br")
    print("Transcrição:", texto)
except sr.UnknownValueError:
    print("Não entendi o áudio. Verifique se há fala clara no arquivo.")
except sr.RequestError as e:
    print("Erro ao acessar a API do Google:", e)

In [ ]:
from deep_translator import MyMemoryTranslator


idiomas = {
    "1": {
        "nome": "Inglês",
        "codigo": "english"
    },
    "2": {
        "nome": "Espanhol",
        "codigo": "spanish"
    },
    "3": {
        "nome": "Francês",
        "codigo": "french"
    },
    "4": {
        "nome": "Italiano",
        "codigo": "italian"
    },
    "5": {
        "nome": "Alemão",
        "codigo": "german"
    },
    "6": {
        "nome": "Portugues",
        "codigo": "portuguese"
    }
}


if texto:
    print("Texto original:")
    print(texto)

    print("\nEscolha o idioma da tradução:")
    print("1 - Inglês")
    print("2 - Espanhol")
    print("3 - Francês")
    print("4 - Italiano")
    print("5 - Alemão")

    opcao = input("\nDigite uma opção: ")

    if opcao in idiomas:
        idioma = idiomas[opcao]

        try:
            traducao = MyMemoryTranslator(
                source="english",
                target=idioma["codigo"]
            ).translate(texto)

            print(f"\nTradução para {idioma['nome']}:")
            print(traducao)

        except Exception as erro:
            print(f"Não foi possível realizar a tradução: {erro}")

    else:
        print("Opção de idioma inválida.")

else:
    print("Primeiro grave e transcreva uma fala.")